# Quantum barriers, wells, and resonant tunnelling

This Stage 2 notebook studies finite onsite profiles in a one-dimensional tight-binding device. It uses the functions in `src/quantum_transport` and the saved high-resolution sweep from the Stage 2 analysis. A large transmission peak alone is not used to identify a well resonance; the LDOS is examined as well.

In [ ]:
import csv
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt
import numpy as np
from quantum_transport import (
    clean_potential,
    finite_well_potential,
    left_broadening_matrix,
    local_density_of_states,
    retarded_device_green_function,
    right_broadening_matrix,
    single_barrier_potential,
    symmetric_double_barrier_potential,
    surface_green_function,
    total_density_of_states,
    tight_binding_hamiltonian,
    lead_self_energy,
    landauer_transmission,
)

N, t, epsilon_0, t_c = 60, 1.0, 0.0, 1.0
barrier_height, barrier_width = 1.5, 3
well_depth, well_width = 0.8, 14
site = np.arange(N)

## 1. Device profiles

The clean chain has uniform onsite energy. The single barrier raises one site by 1.5. The finite well lowers sites 23 through 36 by 0.8. The double barrier is centered and symmetric, with two three-site barriers around a 14-site well.

In [ ]:
V_clean = clean_potential(N, epsilon_0)
V_barrier = single_barrier_potential(N, epsilon_0, barrier_height, site_index=29)
V_well = finite_well_potential(N, epsilon_0, well_depth, start_index=23, stop_index=37)
V_double = symmetric_double_barrier_potential(
    N, epsilon_0, barrier_height, well_depth, barrier_width, well_width
)
np.array_equal(V_double, V_double[::-1]), V_double.shape

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.step(site, V_clean, where="mid", label="Clean", color="black")
ax.step(site, V_barrier, where="mid", label="Single barrier")
ax.step(site, V_double, where="mid", label="Symmetric double barrier")
ax.set(xlabel="Site index (zero-based)", ylabel="Onsite energy (units of t)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig

## 2. Lead band, surface Green function, and self-energy

The identical ideal leads have dispersion `E(k) = epsilon_0 - 2 t cos(k)` and band `[-2t, 2t]`. The retarded surface Green function selects the negative-imaginary branch inside this band; the endpoint self-energy is `Sigma^r = t_c^2 g_s^r`.

In [ ]:
band = (epsilon_0 - 2 * t, epsilon_0 + 2 * t)
E_sample = -0.73095
g_surface = surface_green_function(E_sample, epsilon_0, t)
sigma = lead_self_energy(E_sample, epsilon_0, t, t_c)
band, g_surface, sigma

## 3. Transmission spectra

The high-resolution Stage 2 sweep is loaded from the numerical output generated by the source functions. It compares clean, single-barrier, finite-well, and double-barrier devices without normalizing any curve.

In [ ]:
sweep = np.genfromtxt(project_root / "results/stage2_transmission.csv", delimiter=",", names=True)
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
for key, label in [("clean", "Clean"), ("single_barrier", "Single barrier"), ("finite_well", "Finite well"), ("double_barrier", "Double barrier")]:
    ax.plot(sweep["energy"], sweep[key], label=label)
ax.axvline(band[0], color="0.5", linestyle="--", linewidth=0.8)
ax.axvline(band[1], color="0.5", linestyle="--", linewidth=0.8)
ax.set(xlabel="Energy E (units of t)", ylabel="Transmission T(E)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig

## 4. Resonance selection and Green functions

The recorded primary LDOS resonance is the highest-transmission detected peak below the barrier-region band edge. That energy region is evanescent in the barrier material. We then independently inspect whether the LDOS is concentrated in the central well.

In [ ]:
peak_table = np.genfromtxt(project_root / "results/stage2_resonances.csv", delimiter=",", names=True)
barrier_band_lower_edge = epsilon_0 + barrier_height - 2 * t
barrier_candidates = peak_table[peak_table["energy"] < barrier_band_lower_edge]
peak = barrier_candidates[np.argmax(barrier_candidates["transmission"])]
E_res = float(peak["energy"])
H_double = tight_binding_hamiltonian(
    N, V_double, t
)
sigma_res = lead_self_energy(E_res, epsilon_0, t, t_c)
G_res = retarded_device_green_function(E_res, H_double, sigma_res, sigma_res)
Gamma_L = left_broadening_matrix(N, sigma_res)
Gamma_R = right_broadening_matrix(N, sigma_res)
T_res = landauer_transmission(G_res, Gamma_L, Gamma_R)
peak, T_res

## 5. Total DOS

The total DOS is computed as `-Im(Tr(G^r))/pi`; it should equal the sum of site LDOS values at each energy.

In [ ]:
dos_data = np.genfromtxt(project_root / "results/stage2_dos.csv", delimiter=",", names=True)
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(dos_data["energy"], dos_data["total_dos_from_trace"], color="#533a71")
ax.set(xlabel="Energy E (units of t)", ylabel="Total DOS (1 / energy unit)")
ax.grid(alpha=0.2)
fig

## 6. Spatial LDOS through the resonance

The LDOS is `rho_i(E) = -Im(G^r_ii(E))/pi`. The three profiles below are evaluated just below, at, and just above the selected peak. The reported central-well fraction compares the sum over sites 23?36 with the total DOS.

In [ ]:
detuning = 0.04
energies_near_peak = [E_res - detuning, E_res, E_res + detuning]
ldos_near_peak = []
t_values_near_peak = []
for energy in energies_near_peak:
    sigma_e = lead_self_energy(energy, epsilon_0, t, t_c)
    G_e = retarded_device_green_function(energy, H_double, sigma_e, sigma_e)
    ldos_e = local_density_of_states(G_e)
    total_e = total_density_of_states(G_e)
    ldos_near_peak.append(ldos_e)
    t_values_near_peak.append(landauer_transmission(
        G_e,
        left_broadening_matrix(N, sigma_e),
        right_broadening_matrix(N, sigma_e),
    ))
well_fraction = ldos_near_peak[1][23:37].sum() / ldos_near_peak[1].sum()
list(zip(energies_near_peak, t_values_near_peak)), well_fraction

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
for energy, density in zip(energies_near_peak, ldos_near_peak):
    ax.plot(site, density, label=f"E={energy:.4f}")
ax.axvspan(22.5, 36.5, color="#d9cce8", alpha=0.25, label="Central well")
ax.set(xlabel="Site index (zero-based)", ylabel="Local DOS (1 / energy unit)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig

At the selected finite-device resonance, the computed transmission is near one and the central well contains most of the total DOS. The nearby detuned LDOS profiles test whether that spatial concentration is energy-selective. These results describe this finite coherent model and are not a universal resonance energy or an experimental comparison.